In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import kagglehub

# Download latest version
DATASET_PATH = kagglehub.dataset_download("joebeachcapital/windows-malwares")

print("Dataset downloaded to:")
print(DATASET_PATH)

Mounting files to /kaggle/input/datasets/joebeachcapital/windows-malwares...
Dataset downloaded to:
/kaggle/input/datasets/joebeachcapital/windows-malwares


In [3]:
import os

print("Files in dataset:\n")

for file in sorted(os.listdir(DATASET_PATH)):
    print(file)

Files in dataset:

API_Functions.csv
DLLs_Imported.csv
PE_Header.csv
PE_Section.csv


In [4]:
import os

for file in sorted(os.listdir(DATASET_PATH)):
    path = os.path.join(DATASET_PATH, file)

    size_gb = os.path.getsize(path) / (1024**3)

    print(f"{file:<25} {size_gb:.2f} GB")

API_Functions.csv         1.21 GB
DLLs_Imported.csv         0.04 GB
PE_Header.csv             0.01 GB
PE_Section.csv            0.01 GB


In [8]:
import numpy as np

api_cols = api_df.columns.drop(["SHA256", "Type"])

api_df[api_cols] = api_df[api_cols].astype(np.uint8)

api_df["Type"] = api_df["Type"].astype(np.uint8)

api_df.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29505 entries, 0 to 29504
Columns: 21920 entries, SHA256 to setupdigethwprofilefriendlynameexw
dtypes: object(1), uint8(21919)
memory usage: 619.9 MB


In [9]:
api_df.to_parquet(
    "API_Functions_Optimized.parquet",
    index=False
)

In [1]:
import pandas as pd
import numpy as np

# ==========================
# Load Dataset
# ==========================

dll_df = pd.read_csv(
    "/kaggle/input/datasets/joebeachcapital/windows-malwares/DLLs_Imported.csv"
)

print("="*80)
print("Basic Information")
print("="*80)

print("Shape:", dll_df.shape)

print("\nFirst 5 rows")
display(dll_df.head())

print("\nFirst 20 columns")
print(dll_df.columns[:20])

print("\nLast 20 columns")
print(dll_df.columns[-20:])

print("\nData Types")
print(dll_df.dtypes.value_counts())

print("\nMemory Usage")
dll_df.info(memory_usage="deep")

# ==========================
# Memory Optimization
# ==========================

dll_cols = dll_df.columns.drop(["SHA256", "Type"])

dll_df[dll_cols] = dll_df[dll_cols].astype(np.uint8)
dll_df["Type"] = dll_df["Type"].astype(np.uint8)

print("\nAfter Optimization")
dll_df.info(memory_usage="deep")

# ==========================
# Save Optimized Version
# ==========================

dll_df.to_parquet(
    "DLLs_Imported_Optimized.parquet",
    index=False
)

# ==========================
# Label Distribution
# ==========================

print("\nLabel Distribution")
print(dll_df["Type"].value_counts())

print("\nLabel Proportion")
print(dll_df["Type"].value_counts(normalize=True))

# ==========================
# DLL Frequency
# ==========================

dll_usage = dll_df.drop(columns=["SHA256", "Type"]).sum()

print("\nDLL Usage Statistics")
print(dll_usage.describe())

print("\nTop 30 DLLs")
print(dll_usage.sort_values(ascending=False).head(30))

print("\nLeast Used DLLs")
print(dll_usage.sort_values().head(30))

# ==========================
# Sparsity
# ==========================

print("\nDLLs appearing exactly once:")
print((dll_usage == 1).sum())

print("\nDLLs appearing <5 times:")
print((dll_usage < 5).sum())

print("\nDLLs appearing <10 times:")
print((dll_usage < 10).sum())

# ==========================
# Quality Checks
# ==========================

print("\nUnused DLL columns:")
print((dll_usage == 0).sum())

constant_cols = [
    col for col in dll_df.columns[2:]
    if dll_df[col].nunique() == 1
]

print("\nConstant Columns:")
print(len(constant_cols))

Basic Information
Shape: (29498, 631)

First 5 rows


,SHA256,Type,advapi32.dll,kernel32.dll,vspmsg.dll,ole32.dll,oleaut32.dll,psapi.dll,setupapi.dll,shlwapi.dll,...,odbccp32.dll,api-ms-win-crt-environment-l1-1-0.dll,api-ms-win-core-memory-l1-1-3.dll,api-ms-win-core-datetime-l1-1-0.dll,api-ms-win-core-psapi-ansi-l1-1-0.dll,api-ms-win-core-fibers-l1-1-0.dll,api-ms-win-core-file-l2-1-0.dll,api-ms-win-core-sysinfo-l1-2-0.dll,dbgeng.dll,d3d11.dll
0,002ce0d28ec990aadbbc89df457189de37d8adaadc9c08...,0,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2a053f32b1d48539e3e2807f86754be87ce95b08378467...,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2f031a1752f7067fb9f483ae0ac5f3036c9b66cc4af40e...,0,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,308e8bb2e8a3b67607d2454370e0b50147b42049bda813...,0,1,1,0,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,31aaba443b9869e6e68c17125f8d7989cbd762fb38ae3a...,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



First 20 columns
Index(['SHA256', 'Type', 'advapi32.dll', 'kernel32.dll', 'vspmsg.dll',
       'ole32.dll', 'oleaut32.dll', 'psapi.dll', 'setupapi.dll', 'shlwapi.dll',
       'pdh.dll', 'xmllite.dll', 'msvcr110.dll', 'user32.dll', 'msvcrt.dll',
       'shell32.dll', 'ntdll.dll', 'api-ms-win-core-winrt-l1-1-0.dll',
       'dui70.dll', 'windows.ui.immersive.dll'],
      dtype='object')

Last 20 columns
Index(['api-ms-win-core-console-l3-2-0.dll',
       'api-ms-win-core-localization-l1-2-0.dll',
       'api-ms-win-core-errorhandling-l1-1-0.dll',
       'api-ms-win-core-processthreads-l1-1-1.dll',
       'api-ms-win-core-interlocked-l1-1-0.dll',
       'api-ms-win-core-debug-l1-1-0.dll',
       'api-ms-win-core-rtlsupport-l1-1-0.dll',
       'api-ms-win-core-file-l1-1-0.dll', 'api-ms-win-core-heap-l1-1-0.dll',
       'api-ms-win-ntuser-sysparams-l1-1-0.dll', 'odbccp32.dll',
       'api-ms-win-crt-environment-l1-1-0.dll',
       'api-ms-win-core-memory-l1-1-3.dll',
       'api-ms-win-core

In [2]:
import pandas as pd
import numpy as np

# ==========================
# Load Dataset
# ==========================

pe_df = pd.read_csv(
    "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Header.csv"
)

print("="*80)
print("Basic Information")
print("="*80)

print("Shape:", pe_df.shape)

display(pe_df.head())

print("\nColumns")
print(pe_df.columns.tolist())

print("\nData Types")
print(pe_df.dtypes.value_counts())

print("\nMemory Usage")
pe_df.info(memory_usage="deep")

# ==========================
# Optimize
# ==========================

# Optimize only the label
pe_df["Type"] = pe_df["Type"].astype(np.uint8)

print("\nAfter Optimization")
pe_df.info(memory_usage="deep")

# Save optimized version
pe_df.to_parquet(
    "PE_Header_Optimized.parquet",
    index=False
)

# ==========================
# Label Distribution
# ==========================

print("\nLabel Distribution")
print(pe_df["Type"].value_counts())

# ==========================
# Missing Values
# ==========================

print("\nMissing Values")
print(pe_df.isnull().sum().sort_values(ascending=False).head(20))

# ==========================
# Constant Columns
# ==========================

constant_cols = []

for col in pe_df.columns[2:]:
    if pe_df[col].nunique() == 1:
        constant_cols.append(col)

print("\nConstant Columns")
print(len(constant_cols))
print(constant_cols)

# ==========================
# Numerical Summary
# ==========================

print("\nStatistical Summary")
display(pe_df.describe())

Basic Information
Shape: (29807, 54)


,SHA256,Type,e_magic,e_cblp,e_cp,e_crlc,e_cparhdr,e_minalloc,e_maxalloc,e_ss,...,SizeOfImage,SizeOfHeaders,CheckSum,Subsystem,DllCharacteristics,SizeOfStackReserve,SizeOfHeapReserve,SizeOfHeapCommit,LoaderFlags,NumberOfRvaAndSizes
0,dacbe8cb72dd746539792a50e84965fefef73feaa07b5d...,0,23117,144,3,0,4,0,65535,0,...,139264,4096,0,2,1344,1048576,1048576,4096,0,16
1,d3dc7512ce75db33b2c3063fa99245e9ca9fe3b086462f...,0,23117,144,3,0,4,0,65535,0,...,49152,512,0,2,34112,1048576,1048576,4096,0,16
2,b350fac81533f02981dc2176ed17163177d92d9405758e...,0,23117,144,3,0,4,0,65535,0,...,532480,512,0,2,34144,1048576,1048576,4096,0,16
3,dfee618043a47b7b09305df0ca460559d9f567ee246c7b...,0,23117,144,3,0,4,0,65535,0,...,1368064,4096,1366781,2,1024,1048576,1048576,4096,0,16
4,c7b2e4e4fb2fcc44c953673ff57c3d14bdf5d2008f35e9...,0,23117,144,3,0,4,0,65535,0,...,32768,512,64362,2,1344,1048576,1048576,4096,0,16



Columns
['SHA256', 'Type', 'e_magic', 'e_cblp', 'e_cp', 'e_crlc', 'e_cparhdr', 'e_minalloc', 'e_maxalloc', 'e_ss', 'e_sp', 'e_csum', 'e_ip', 'e_cs', 'e_lfarlc', 'e_ovno', 'e_oemid', 'e_oeminfo', 'e_lfanew', 'Machine', 'NumberOfSections', 'TimeDateStamp', 'PointerToSymbolTable', 'NumberOfSymbols', 'SizeOfOptionalHeader', 'Characteristics', 'Magic', 'MajorLinkerVersion', 'MinorLinkerVersion', 'SizeOfCode', 'SizeOfInitializedData', 'SizeOfUninitializedData', 'AddressOfEntryPoint', 'BaseOfCode', 'ImageBase', 'SectionAlignment', 'FileAlignment', 'MajorOperatingSystemVersion', 'MinorOperatingSystemVersion', 'MajorImageVersion', 'MinorImageVersion', 'MajorSubsystemVersion', 'MinorSubsystemVersion', 'Reserved1', 'SizeOfImage', 'SizeOfHeaders', 'CheckSum', 'Subsystem', 'DllCharacteristics', 'SizeOfStackReserve', 'SizeOfHeapReserve', 'SizeOfHeapCommit', 'LoaderFlags', 'NumberOfRvaAndSizes']

Data Types
int64     53
object     1
Name: count, dtype: int64

Memory Usage
<class 'pandas.core.frame.D

,Type,e_magic,e_cblp,e_cp,e_crlc,e_cparhdr,e_minalloc,e_maxalloc,e_ss,e_sp,...,SizeOfImage,SizeOfHeaders,CheckSum,Subsystem,DllCharacteristics,SizeOfStackReserve,SizeOfHeapReserve,SizeOfHeapCommit,LoaderFlags,NumberOfRvaAndSizes
count,29807.000000,29807.0,29807.000000,29807.000000,29807.000000,29807.000000,29807.000000,29807.000000,29807.000000,29807.000000,...,2.980700e+04,29807.000000,2.980700e+04,29807.000000,29807.000000,2.980700e+04,2.980700e+04,29807.000000,29807.0,29807.000000
mean,3.137719,23117.0,251.805415,754.102593,404.025464,43.835576,354.468078,64788.852551,198.975778,378.190425,...,1.867389e+06,1452.845305,1.106016e+06,2.077901,23421.903177,1.090262e+06,1.063879e+06,4096.420136,0.0,15.991445
std,1.801958,0.0,1532.495681,6583.983720,3640.873340,1290.672177,3120.411120,6421.027708,1810.405076,2324.751433,...,6.605455e+06,1365.458901,4.201892e+07,0.268020,15721.665448,5.435179e+05,2.304633e+05,245.286161,0.0,0.101127
min,0.000000,23117.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.048000e+04,512.000000,0.000000e+00,2.000000,0.000000,0.000000e+00,8.850000e+02,0.000000,0.0,14.000000
25%,2.000000,23117.0,144.000000,3.000000,0.000000,4.000000,0.000000,65535.000000,0.000000,184.000000,...,2.457600e+05,512.000000,0.000000e+00,2.000000,0.000000,1.048576e+06,1.048576e+06,4096.000000,0.0,16.000000
50%,3.000000,23117.0,144.000000,3.000000,0.000000,4.000000,0.000000,65535.000000,0.000000,184.000000,...,4.997120e+05,1024.000000,0.000000e+00,2.000000,33024.000000,1.048576e+06,1.048576e+06,4096.000000,0.0,16.000000
75%,5.000000,23117.0,144.000000,3.000000,0.000000,4.000000,0.000000,65535.000000,0.000000,184.000000,...,9.441280e+05,1024.000000,3.386080e+05,2.000000,34112.000000,1.048576e+06,1.048576e+06,4096.000000,0.0,16.000000
max,6.000000,23117.0,64110.000000,62388.000000,63030.000000,64913.000000,57879.000000,65535.000000,59756.000000,64539.000000,...,1.225933e+08,8192.000000,3.885444e+09,3.000000,50496.000000,6.710886e+07,4.194304e+06,16384.000000,0.0,16.000000


In [3]:
import pandas as pd
import numpy as np

# ==========================
# Load Dataset
# ==========================

section_df = pd.read_csv(
    "/kaggle/input/datasets/joebeachcapital/windows-malwares/PE_Section.csv"
)

print("="*80)
print("Basic Information")
print("="*80)

print("Shape:", section_df.shape)

display(section_df.head())

print("\nColumns")
print(section_df.columns.tolist())

print("\nData Types")
print(section_df.dtypes.value_counts())

print("\nMemory Usage")
section_df.info(memory_usage="deep")

# ==========================
# Optimize
# ==========================

section_df["Type"] = section_df["Type"].astype(np.uint8)

print("\nAfter Optimization")
section_df.info(memory_usage="deep")

section_df.to_parquet(
    "PE_Section_Optimized.parquet",
    index=False
)

# ==========================
# Label Distribution
# ==========================

print("\nLabel Distribution")
print(section_df["Type"].value_counts())

# ==========================
# Missing Values
# ==========================

print("\nTop Missing Values")
print(section_df.isnull().sum().sort_values(ascending=False).head(20))

# ==========================
# Constant Columns
# ==========================

constant_cols = [
    col for col in section_df.columns[2:]
    if section_df[col].nunique() == 1
]

print("\nConstant Columns")
print(len(constant_cols))
print(constant_cols)

# ==========================
# Numerical Summary
# ==========================

display(section_df.describe())

Basic Information
Shape: (29760, 92)


,SHA256,Type,text_Misc_VirtualSize,text_VirtualAddress,text_SizeOfRawData,text_PointerToRawData,text_PointerToRelocations,text_PointerToLinenumbers,text_NumberOfRelocations,text_NumberOfLinenumbers,...,tls_Characteristics,pdata_Misc_VirtualSize,pdata_VirtualAddress,pdata_SizeOfRawData,pdata_PointerToRawData,pdata_PointerToRelocations,pdata_PointerToLinenumbers,pdata_NumberOfRelocations,pdata_NumberOfLinenumbers,pdata_Characteristics
0,dacbe8cb72dd746539792a50e84965fefef73feaa07b5d...,0,114580,8192,114688,4096,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,d3dc7512ce75db33b2c3063fa99245e9ca9fe3b086462f...,0,16436,8192,16896,512,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,b350fac81533f02981dc2176ed17163177d92d9405758e...,0,506420,8192,506880,512,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,dfee618043a47b7b09305df0ca460559d9f567ee246c7b...,0,1312036,8192,1314816,4096,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,c7b2e4e4fb2fcc44c953673ff57c3d14bdf5d2008f35e9...,0,2660,8192,3072,512,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0



Columns
['SHA256', 'Type', 'text_Misc_VirtualSize', 'text_VirtualAddress', 'text_SizeOfRawData', 'text_PointerToRawData', 'text_PointerToRelocations', 'text_PointerToLinenumbers', 'text_NumberOfRelocations', 'text_NumberOfLinenumbers', 'text_Characteristics', 'data_Misc_VirtualSize', 'data_VirtualAddress', 'data_SizeOfRawData', 'data_PointerToRawData', 'data_PointerToRelocations', 'data_PointerToLinenumbers', 'data_NumberOfRelocations', 'data_NumberOfLinenumbers', 'data_Characteristics', 'rdata_Misc_VirtualSize', 'rdata_VirtualAddress', 'rdata_SizeOfRawData', 'rdata_PointerToRawData', 'rdata_PointerToRelocations', 'rdata_PointerToLinenumbers', 'rdata_NumberOfRelocations', 'rdata_NumberOfLinenumbers', 'rdata_Characteristics', 'bss_Misc_VirtualSize', 'bss_VirtualAddress', 'bss_SizeOfRawData', 'bss_PointerToRawData', 'bss_PointerToRelocations', 'bss_PointerToLinenumbers', 'bss_NumberOfRelocations', 'bss_NumberOfLinenumbers', 'bss_Characteristics', 'idata_Misc_VirtualSize', 'idata_Virtual

,Type,text_Misc_VirtualSize,text_VirtualAddress,text_SizeOfRawData,text_PointerToRawData,text_PointerToRelocations,text_PointerToLinenumbers,text_NumberOfRelocations,text_NumberOfLinenumbers,text_Characteristics,...,tls_Characteristics,pdata_Misc_VirtualSize,pdata_VirtualAddress,pdata_SizeOfRawData,pdata_PointerToRawData,pdata_PointerToRelocations,pdata_PointerToLinenumbers,pdata_NumberOfRelocations,pdata_NumberOfLinenumbers,pdata_Characteristics
count,29760.000000,2.976000e+04,2.976000e+04,2.976000e+04,2.976000e+04,2.976000e+04,2.976000e+04,29760.000000,29760.000000,2.976000e+04,...,2.976000e+04,2.976000e+04,2.976000e+04,29760.000000,2.976000e+04,29760.0,29760.0,29760.0,29760.0,2.976000e+04
mean,3.136223,3.330184e+05,9.844301e+03,3.320933e+05,4.487639e+03,2.054643e+05,5.017786e+04,1.837399,1.932695,1.462997e+09,...,1.257468e+08,3.634925e+02,4.205557e+03,126.018683,3.711226e+03,0.0,0.0,0.0,0.0,2.416933e+07
std,1.802487,6.750130e+05,8.736727e+04,6.737745e+05,5.991141e+04,2.675110e+07,8.584756e+06,227.523118,213.910180,4.933275e+08,...,6.243773e+08,2.011117e+04,9.049542e+04,3535.954393,8.777450e+04,0.0,0.0,0.0,0.0,1.707008e+08
min,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000e+00
25%,2.000000,3.102400e+04,4.096000e+03,3.020800e+04,5.120000e+02,0.000000e+00,0.000000e+00,0.000000,0.000000,1.610613e+09,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000e+00
50%,3.000000,1.496090e+05,4.096000e+03,1.495040e+05,1.024000e+03,0.000000e+00,0.000000e+00,0.000000,0.000000,1.610613e+09,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000e+00
75%,5.000000,4.873300e+05,8.192000e+03,4.843520e+05,1.024000e+03,0.000000e+00,0.000000e+00,0.000000,0.000000,1.610613e+09,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.000000e+00
max,6.000000,3.379818e+07,8.036352e+06,3.379866e+07,8.022528e+06,4.261479e+09,1.480939e+09,34048.000000,30144.000000,3.763339e+09,...,3.758096e+09,3.284992e+06,7.737344e+06,509025.000000,7.721984e+06,0.0,0.0,0.0,0.0,3.758096e+09
